# Análisis de sentimiento de reseñas de clientes

**Objetivo de negocio.** Comprobar si el sentimiento escrito en las 500 reseñas coincide con el promedio humano de 4.5/5 estrellas. El resultado debe indicar cuántas reseñas son positivas, neutrales o negativas, y señalar posibles falsos negativos del modelo.

In [1]:
# Exploramos la calidad y estructura antes de usar un modelo.
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'src' else Path.cwd()
DATA_PATH = PROJECT_ROOT / 'data' / 'raw' / 'reviews.csv'
reviews = pd.read_csv(DATA_PATH)
print(f'Filas y columnas: {reviews.shape}')
print('\nTipos de datos:')
print(reviews.dtypes)
print('\nValores faltantes:')
print(reviews.isna().sum())
print('\nDuplicados de review_id:', reviews['review_id'].duplicated().sum())
reviews.head(3)

Filas y columnas: (500, 3)

Tipos de datos:
review_id      int64
rating         int64
review_text      str
dtype: object

Valores faltantes:
review_id      0
rating         0
review_text    0
dtype: int64

Duplicados de review_id: 0


,review_id,rating,review_text
0,1,5,Visited Harbor House Café last weekend. Great ...
1,2,5,Tried Harbor House Café after seeing it recomm...
2,3,5,Tried Harbor House Café after seeing it recomm...


## EDA inicial

El archivo contiene 500 reseñas y las tres columnas esperadas. Revisamos la distribución de estrellas y la longitud del texto.

In [2]:
# La distribución humana es la referencia para comparar el modelo.
reviews['text_length'] = reviews['review_text'].str.len()
print('Distribución de ratings:')
print(reviews['rating'].value_counts().sort_index())
print('\nPromedio humano:', round(reviews['rating'].mean(), 2))
print('\nEstadísticas de longitud:')
print(reviews['text_length'].describe().round(2))
print('\nMuestra:')
for text in reviews['review_text'].head(3): print('-', text)

Distribución de ratings:
rating
1     12
2     18
3     38
4     72
5    360
Name: count, dtype: int64

Promedio humano: 4.5

Estadísticas de longitud:
count    500.00
mean     170.17
std       27.06
min       86.00
25%      155.00
50%      173.00
75%      189.00
max      230.00
Name: text_length, dtype: float64

Muestra:
- Visited Harbor House Café last weekend. Great spot to work or catch up with friends. Every dish was bursting with flavor. Worth every penny. Already planning my next visit.
- Tried Harbor House Café after seeing it recommended online. The coffee was rich and perfectly brewed. Great spot to work or catch up with friends. Great value for what you get. Five stars without hesitation.
- Tried Harbor House Café after seeing it recommended online. Great spot to work or catch up with friends. The seasonal menu never disappoints. Great value for what you get. Already planning my next visit.


## Insights, limpieza y plan

Los datos están completos, no tienen IDs duplicados y las reseñas tienen texto suficiente. La puntuación humana está concentrada en 5 estrellas: 360 de 500. No hace falta eliminar filas ni imputar valores; la longitud se usa solo para EDA.

Usaremos `nlptown/bert-base-multilingual-uncased-sentiment`, fijado al commit `8f6f4e3a8f70be4b65d3a4a8762b6d781cda240d`. El modelo predice 1–5 estrellas: 1–2 serán **negativo**, 3 **neutral** y 4–5 **positivo**. Se cargará una sola vez.

In [3]:
# Cargamos el modelo una vez y lo reutilizamos para todas las reseñas.
from transformers import pipeline
MODEL_NAME = 'nlptown/bert-base-multilingual-uncased-sentiment'
MODEL_REVISION = '8f6f4e3a8f70be4b65d3a4a8762b6d781cda240d'
classifier = pipeline('sentiment-analysis', model=MODEL_NAME, revision=MODEL_REVISION)
predictions = classifier(reviews['review_text'].tolist(), truncation=True, batch_size=16)
reviews['predicted_stars'] = [int(item['label'].split()[0]) for item in predictions]
reviews['prediction_confidence'] = [item['score'] for item in predictions]
reviews['sentiment_band'] = reviews['predicted_stars'].map(lambda x: 'negativo' if x <= 2 else 'neutral' if x == 3 else 'positivo')
reviews[['review_id','rating','predicted_stars','sentiment_band']].head()

config.json:   0%|          | 0.00/953 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/669M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/39.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Device set to use cpu


,review_id,rating,predicted_stars,sentiment_band
0,1,5,5,positivo
1,2,5,5,positivo
2,3,5,5,positivo
3,4,5,5,positivo
4,5,5,5,positivo


## Distribución predicha y comparación

Calculamos porcentajes y los contrastamos con el promedio humano de 4.5/5.

In [4]:
# Resumen de las bandas de sentimiento.
sentiment_counts = reviews['sentiment_band'].value_counts().reindex(['positivo','neutral','negativo'], fill_value=0)
comparison = pd.DataFrame({'cantidad': sentiment_counts, 'porcentaje': (sentiment_counts / len(reviews) * 100).round(2)})
print(comparison)
print('\nPromedio humano:', round(reviews['rating'].mean(), 2), '/ 5')
print('Promedio predicho:', round(reviews['predicted_stars'].mean(), 2), '/ 5')
print('\nCruce:')
print(pd.crosstab(reviews['rating'], reviews['sentiment_band']))

                cantidad  porcentaje
sentiment_band                      
positivo             412        82.4
neutral               41         8.2
negativo              47         9.4

Promedio humano: 4.5 / 5
Promedio predicho: 4.4 / 5

Cruce:
sentiment_band  negativo  neutral  positivo
rating                                     
1                     12        0         0
2                     16        2         0
3                      3       35         0
4                      1        3        68
5                     15        1       344


## Falsos negativos y revisión manual

Un falso negativo es una reseña con rating humano 4–5 que el modelo clasifica con 1–2 estrellas. También revisamos manualmente 20 reseñas.

In [5]:
# Identificamos falsos negativos frente a la puntuación humana.
false_negatives = reviews[(reviews['rating'] >= 4) & (reviews['predicted_stars'] <= 2)].copy()
print('Cantidad de falsos negativos:', len(false_negatives))
false_negatives[['review_id','rating','predicted_stars','prediction_confidence','review_text']].head(10)

Cantidad de falsos negativos: 16


,review_id,rating,predicted_stars,prediction_confidence,review_text
13,14,4,1,0.444348,Tried Harbor House Café after seeing it recomm...
14,15,5,1,0.389345,Every dish was bursting with flavor. The place...
72,73,5,1,0.267875,Stopped by Harbor House Café for the first tim...
86,87,5,1,0.324704,Tried Harbor House Café after seeing it recomm...
137,138,5,2,0.275117,Tried Harbor House Café after seeing it recomm...
199,200,5,2,0.368520,Stopped by Harbor House Café for the first tim...
203,204,5,1,0.349168,Came to Harbor House Café for a birthday brunc...
253,254,5,1,0.364092,Stopped by Harbor House Café for the first tim...
269,270,5,1,0.409077,Stopped by Harbor House Café for the first tim...
321,322,5,1,0.607707,Every dish was bursting with flavor. The staff...


## Muestra manual de 20 reseñas

La selección es fija para que la revisión sea reproducible.

In [6]:
# Inspeccionamos una muestra reproducible.
manual_sample = reviews.sample(n=20, random_state=42)[['review_id','rating','predicted_stars','sentiment_band','review_text']].sort_values('review_id')
manual_sample

,review_id,rating,predicted_stars,sentiment_band,review_text
9,10,5,5,positivo,The pastries were incredible. The staff seemed...
30,31,5,5,positivo,Grabbed a quick coffee at Harbor House Café th...
68,69,5,5,positivo,Visited Harbor House Café last weekend. The pl...
73,74,5,5,positivo,Stopped by Harbor House Café for the first tim...
84,85,5,4,positivo,Tried Harbor House Café after seeing it recomm...
104,105,5,5,positivo,Grabbed a quick coffee at Harbor House Café th...
124,125,5,5,positivo,Regular customer at Harbor House Café here. Th...
155,156,4,4,positivo,Grabbed a quick coffee at Harbor House Café th...
194,195,2,1,negativo,Grabbed a quick coffee at Harbor House Café th...
316,317,5,5,positivo,Best brunch i've had in months. Nobody checked...


**Notas de revisión manual:**

- La reseña `409` tiene rating humano de 5 estrellas y describe comida, servicio y una visita futura, pero el modelo predice 1 estrella: es un falso negativo claro.
- Las reseñas `10`, `156` y `496` mencionan personal molesto, pero mantienen una valoración humana alta porque la comida compensa el problema; el modelo las clasifica como positivas, una decisión razonable.
- Las reseñas `362`, `407` y `451` expresan una experiencia promedio y fueron clasificadas como neutrales, por lo que estas predicciones son coherentes.
- La muestra confirma que el modelo suele reconocer el sentimiento general, pero puede fallar cuando aparecen frases positivas repetidas o cuando el dominio de servicio difiere del de productos.

## Conclusiones

El promedio humano de 4.5/5 indica una experiencia globalmente positiva. La distribución predicha cuantifica cómo interpreta el texto el modelo, pero no es perfecta: fue entrenado principalmente con reseñas de productos y aquí analizamos servicios. Los errores esperables incluyen lenguaje indirecto, problemas mencionados dentro de reseñas positivas y referencias a personal, espera o ambiente.

**Recomendación:** usar el modelo como primera clasificación y acompañar el reporte con distribución, promedio humano y ejemplos de falsos negativos. No presentar una predicción negativa como verdad definitiva sin revisión humana.

## Exportación productiva

La lógica limpia también está en `src/app.py`. Guardamos el CSV enriquecido solicitado.

In [7]:
# Exportamos sin la columna auxiliar usada solo para EDA.
output_path = PROJECT_ROOT / 'data' / 'processed' / 'reviews_with_sentiment.csv'
reviews.drop(columns=['text_length']).to_csv(output_path, index=False)
print(f'Archivo guardado en: {output_path}')
print(f'Filas guardadas: {len(reviews)}')

Archivo guardado en: /workspaces/jesteban1983-WeLoveReviews/data/processed/reviews_with_sentiment.csv
Filas guardadas: 500
